# Comparación de Modelos de Clasificación Binaria con Datos Reales

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn --quiet

## Objetivo del Notebook

Este notebook tiene como objetivo demostrar el proceso completo de construcción y comparación de modelos de machine learning para un problema de clasificación binaria utilizando datos reales. Específicamente, utilizaremos el dataset **Breast Cancer Wisconsin** para clasificar tumores de mama como benignos o malignos.

El flujo de trabajo incluye:
1. **Carga y exploración de datos**: Importar el dataset y entender su estructura.
2. **Preprocesamiento**: Dividir los datos en entrenamiento y prueba, y normalizar las características.
3. **Entrenamiento de modelos**: Implementar y entrenar dos algoritmos diferentes (Regresión Logística y Árbol de Decisión).
4. **Evaluación**: Calcular métricas de rendimiento y visualizar curvas ROC.
5. **Comparación**: Analizar los resultados para determinar cuál modelo es más adecuado.

Este ejercicio es fundamental para entender conceptos clave en machine learning como sobreajuste, evaluación de modelos y selección de algoritmos.

## Uso de Datos Reales
En esta sección utilizaremos el dataset **Breast Cancer Wisconsin** de `sklearn.datasets`, que contiene información sobre tumores de mama para clasificar si son benignos o malignos.

### Detalles del Dataset Breast Cancer Wisconsin

El dataset **Breast Cancer Wisconsin (Diagnostic)** es un conjunto de datos clásico en machine learning, disponible en la librería scikit-learn. Fue creado por el Dr. William H. Wolberg de la Universidad de Wisconsin y contiene mediciones de características celulares de biopsias de mama.

**Características principales:**
- **Número de muestras**: 569 (212 malignas, 357 benignas)
- **Número de características**: 30 (todas numéricas continuas)
- **Características incluyen**: radio medio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría y dimensión fractal, medidas para el núcleo celular, peor valor y error estándar.

**Variable objetivo (target)**:
- 0: Tumor maligno (cancerígeno)
- 1: Tumor benigno (no cancerígeno)

Este dataset es ideal para clasificación binaria porque:
- Es desbalanceado (más benignos que malignos), lo que refleja escenarios reales en medicina.
- Las características son numéricas y requieren normalización.
- Permite evaluar la capacidad de los modelos para detectar casos positivos (malignos) correctamente, crucial en diagnósticos médicos.

En este notebook, cargamos los datos desde un archivo CSV previamente guardado para mantener consistencia.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    roc_curve, auc, confusion_matrix
)
import seaborn as sns


### Importación de Librerías

En esta sección importamos todas las librerías necesarias para el análisis:

**Librerías principales:**
- `numpy` y `pandas`: Para manipulación de datos y arrays numéricos.
- `matplotlib.pyplot` y `seaborn`: Para visualización de datos y gráficos.
- `sklearn.model_selection.train_test_split`: Para dividir el dataset en conjuntos de entrenamiento y prueba.
- `sklearn.preprocessing.StandardScaler`: Para normalizar las características, crucial para algoritmos sensibles a la escala como la regresión logística.
- `sklearn.linear_model.LogisticRegression`: Implementación del modelo de regresión logística para clasificación binaria.
- `sklearn.tree.DecisionTreeClassifier`: Modelo de árbol de decisión, útil por su interpretabilidad.
- `sklearn.metrics`: Módulo con funciones para calcular métricas de evaluación como accuracy, precision, recall, ROC curve y AUC.

**Por qué estas librerías:**
- Scikit-learn es la librería estándar para machine learning en Python, proporcionando implementaciones eficientes y bien probadas.
- Pandas facilita el manejo de datos tabulares.
- Matplotlib y Seaborn permiten crear visualizaciones informativas para interpretar los resultados.

In [ ]:
#from sklearn.datasets import load_breast_cancer

## Cargar el dataset Breast Cancer Wisconsin
#data = load_breast_cancer()
#X = data.data
#y = data.target

## Convertir a DataFrame para visualización
#df = pd.DataFrame(X, columns=data.feature_names)
#df['target'] = y

#df.to_csv('cancer_mama.csv', index=False)

### Carga del Dataset

**Nota sobre el código comentado:**
El código comentado muestra cómo cargar el dataset directamente desde scikit-learn y guardarlo como CSV. Esto se hace una vez para tener el archivo local. En producción, es mejor trabajar con archivos locales para reproducibilidad.

**Proceso de carga:**
1. Usamos `pd.read_csv()` para cargar el archivo 'cancer_mama.csv' que contiene los datos del Breast Cancer Wisconsin.
2. El dataset incluye 30 características numéricas y la variable objetivo binaria.
3. Mostramos el DataFrame para verificar que los datos se cargaron correctamente y entender la estructura.

**Importancia de la exploración inicial:**
- Verificar tipos de datos
- Identificar valores faltantes
- Entender la distribución de clases
- Confirmar que las características son numéricas y requieren normalización

In [ ]:
df = pd.read_csv('cancer_mama.csv')
df

### Exploración Inicial de Datos

Al mostrar el DataFrame, podemos observar:
- 569 filas (muestras) y 31 columnas (30 características + target)
- Todas las características son numéricas (float64)
- La columna 'target' contiene valores 0 (maligno) o 1 (benigno)
- No hay valores faltantes visibles

**Próximos pasos:**
- Separar características (X) de la variable objetivo (y)
- Dividir en conjuntos de entrenamiento y prueba
- Normalizar las características

Esta exploración inicial es crucial para entender los datos antes de aplicar cualquier modelo de machine learning.

In [ ]:
# Separar características (X) de la variable objetivo (y)
X = df.drop('target', axis=1)  # Todas las columnas excepto 'target'
y = df['target']  # La columna 'target' (0: maligno, 1: benigno)

print(f"Dimensiones de X: {X.shape}")
print(f"Dimensiones de y: {y.shape}")
print(f"Distribución de clases: {y.value_counts()}")

## División en Conjuntos de Entrenamiento y Prueba
Dividimos los datos en conjuntos de entrenamiento (80%) y prueba (20%) para evaluar los modelos.

In [ ]:
# Dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Importancia de la División Train/Test

**¿Por qué dividir los datos?**
- **Evitar sobreajuste**: Si evaluamos el modelo con los mismos datos usados para entrenar, obtenemos métricas optimistas irreales.
- **Simular escenario real**: En producción, el modelo verá datos nuevos, por lo que debemos evaluar su capacidad de generalización.

**Parámetros utilizados:**
- `test_size=0.2`: 20% para prueba, 80% para entrenamiento. Proporción común que balancea sesgo y varianza.
- `random_state=42`: Semilla para reproducibilidad. Garantiza que la división sea la misma en cada ejecución.

**Resultado esperado:**
- X_train, y_train: Datos para entrenar el modelo
- X_test, y_test: Datos "invisibles" para evaluar el rendimiento final

Esta división es fundamental para obtener una evaluación honesta del modelo.

## Normalización de Datos
Dado que algunos modelos como la regresión logística pueden ser sensibles a la escala de los datos, normalizaremos las características usando `StandardScaler`.

In [ ]:
# Normalizar las características
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### ¿Por qué Normalizar los Datos?

**Problema sin normalización:**
Las características del dataset tienen escalas muy diferentes (ej: área vs. suavidad). Algoritmos basados en distancia o gradientes pueden dar más peso a características con valores más grandes, sesgando el modelo.

**StandardScaler:**
- Resta la media y divide por la desviación estándar
- Resultado: media = 0, desviación estándar = 1
- Preserva la distribución pero la centra en cero

**¿Qué algoritmos requieren normalización?**
- **Regresión Logística**: Muy sensible a la escala, mejora convergencia y rendimiento
- **Árboles de Decisión**: No requieren normalización, ya que son invariantes a escala
- **Buena práctica**: Normalizar siempre, especialmente cuando comparamos modelos

**Proceso:**
- `fit_transform()` en train: Calcula parámetros y transforma
- `transform()` en test: Usa los mismos parámetros (importante para evitar data leakage)

## Entrenamiento de Modelos
Entrenaremos dos modelos de clasificación binaria: 
- **Regresión Logística**
- **Árbol de Decisión** (con profundidad máxima de 3)

In [ ]:
# Entrenar Modelo 1: Regresión Logística
model1 = LogisticRegression()
model1.fit(X_train, y_train)
y_pred1 = model1.predict(X_test)
y_prob1 = model1.predict_proba(X_test)[:, 1]

# Entrenar Modelo 2: Árbol de Decisión
model2 = DecisionTreeClassifier(max_depth=3, random_state=42)
model2.fit(X_train, y_train)
y_pred2 = model2.predict(X_test)
y_prob2 = model2.predict_proba(X_test)[:, 1]

### Modelos Implementados

**1. Regresión Logística:**
- **Tipo**: Modelo lineal para clasificación binaria
- **Cómo funciona**: Estima la probabilidad de que una muestra pertenezca a la clase positiva usando la función sigmoide
- **Ventajas**: Simple, interpretable, eficiente, proporciona probabilidades
- **Parámetros**: Usamos valores por defecto (regularización L2, solver automático)
- **Aplicación**: Ideal para problemas donde necesitamos probabilidades y explicabilidad

**2. Árbol de Decisión:**
- **Tipo**: Modelo no paramétrico basado en reglas
- **Cómo funciona**: Divide el espacio de características en regiones rectangulares basadas en preguntas sí/no
- **Ventajas**: Interpretable (puede visualizarse), maneja relaciones no lineales, no requiere normalización
- **Parámetros**: `max_depth=3` limita la profundidad para evitar sobreajuste
- **Aplicación**: Bueno para datos tabulares, especialmente cuando la interpretabilidad es importante

**Predicciones:**
- `predict()`: Etiquetas de clase (0 o 1)
- `predict_proba()`: Probabilidades para calcular métricas como ROC-AUC

Ambos modelos son apropiados para este problema médico donde la interpretabilidad es crucial.

## Cálculo de Métricas
Calcularemos diferentes métricas para evaluar los modelos:
- Precisión (Accuracy)
- Precisión (Precision)
- Sensibilidad (Recall)
- Especificidad
- AUC-ROC

In [ ]:
# Función para calcular métricas
def calculate_metrics(y_true, y_pred, y_prob):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = auc(fpr, tpr)
    return accuracy, precision, recall, specificity, fpr, tpr, auc_score

# Obtener métricas para ambos modelos
metrics1 = calculate_metrics(y_test, y_pred1, y_prob1)
metrics2 = calculate_metrics(y_test, y_pred2, y_prob2)

### Función de Cálculo de Métricas

Definimos una función `calculate_metrics()` que computa todas las métricas relevantes:

**Métricas calculadas:**
1. **Accuracy (Precisión global)**: Proporción de predicciones correctas. Útil pero puede ser engañosa en datasets desbalanceados.
2. **Precision (Precisión)**: De las predicciones positivas, cuántas son realmente positivas. Importante cuando los falsos positivos son costosos.
3. **Recall (Sensibilidad)**: De los casos positivos reales, cuántos detectamos. Crucial en medicina para no perder diagnósticos.
4. **Specificity (Especificidad)**: De los casos negativos reales, cuántos identificamos correctamente. Complementa al recall.
5. **ROC Curve**: Curva que muestra TPR vs FPR para diferentes umbrales.
6. **AUC**: Área bajo la curva ROC. Mide la capacidad discriminativa del modelo (0.5 = aleatorio, 1.0 = perfecto).

**Matriz de confusión:**
- TN (True Negative): Correctamente identificado como benigno
- FP (False Positive): Falsamente identificado como maligno
- FN (False Negative): Falsamente identificado como benigno (muy grave en medicina)
- TP (True Positive): Correctamente identificado como maligno

En problemas médicos como este, el recall (sensibilidad) es especialmente importante para minimizar falsos negativos.

In [ ]:
# Crear un DataFrame con las métricas
df_metrics = pd.DataFrame({
    "Métrica": ["Precisión (Accuracy)", "Precisión (Precision)", "Sensibilidad (Recall)", "Especificidad", "AUC-ROC"],
    "Regresión Logística": [metrics1[0], metrics1[1], metrics1[2], metrics1[3], metrics1[6]],
    "Árbol de Decisión": [metrics2[0], metrics2[1], metrics2[2], metrics2[3], metrics2[6]]
})

df_metrics

### Interpretación de las Métricas

Al comparar las métricas de ambos modelos, consideramos:

**Regresión Logística vs Árbol de Decisión:**
- **Accuracy**: Ambos modelos muestran buen rendimiento general
- **Precision**: Indica qué tan confiables son las predicciones positivas
- **Recall**: En medicina, es crítico - mide la capacidad de detectar cáncer (malignos)
- **Specificity**: Mide la capacidad de identificar correctamente tumores benignos
- **AUC-ROC**: Métrica robusta, menos sensible al desbalance de clases

**Consideraciones médicas:**
- **Falso negativo (FN)**: Paciente con cáncer no detectado - muy grave
- **Falso positivo (FP)**: Paciente sano diagnosticado con cáncer - genera ansiedad pero es menos grave
- Por esto, el **recall** (sensibilidad) es más importante que la precisión en este contexto

**Análisis esperado:**
- La Regresión Logística suele tener mejor AUC-ROC en datos normalizados
- Los Árboles de Decisión pueden ser más interpretables pero propensos a sobreajuste
- Valores altos en todas las métricas indican buenos modelos para este dataset

## Curva ROC y AUC
La curva ROC muestra la relación entre la tasa de verdaderos positivos (sensibilidad) y la tasa de falsos positivos. El área bajo la curva (AUC) nos indica qué tan bien separa las clases el modelo.

In [ ]:
# Graficar curvas ROC
plt.figure(figsize=(10, 6))
plt.plot(metrics1[4], metrics1[5], label=f"Regresión Logística (AUC = {metrics1[6]:.2f})")
plt.plot(metrics2[4], metrics2[5], label=f"Árbol de Decisión (AUC = {metrics2[6]:.2f})")
plt.plot([0, 1], [0, 1], 'k--', label="Línea Aleatoria")
plt.xlabel("Tasa de Falsos Positivos (FPR)")
plt.ylabel("Tasa de Verdaderos Positivos (TPR)")
plt.title("Curva ROC para ambos modelos")
plt.legend()
plt.show()

### Análisis de la Curva ROC

**¿Qué representa la curva ROC?**
- **Eje X (FPR)**: Tasa de Falsos Positivos = FP / (FP + TN)
- **Eje Y (TPR)**: Tasa de Verdaderos Positivos = TP / (TP + FN) = Recall
- Cada punto representa un umbral de clasificación diferente
- La línea diagonal punteada representa un clasificador aleatorio

**Interpretación del AUC:**
- **AUC = 1.0**: Clasificador perfecto
- **AUC = 0.5**: Clasificador aleatorio (línea diagonal)
- **AUC < 0.5**: Peor que aleatorio (modelo invertido)
- **AUC > 0.8**: Buen clasificador
- **AUC > 0.9**: Excelente clasificador

**Ventajas de ROC-AUC:**
- **Invariante a la escala**: No depende del umbral de clasificación
- **Robusta al desbalance**: Funciona bien cuando las clases no están balanceadas
- **Comparación directa**: Permite comparar modelos fácilmente

**En este contexto médico:**
- Un AUC alto significa que el modelo puede distinguir bien entre tumores benignos y malignos
- La curva por encima de la diagonal indica que ambos modelos son mejores que el azar

## Comparación y Conclusión
Comparando las métricas obtenidas, podemos determinar qué modelo tiene mejor desempeño en la clasificación de tumores benignos y malignos.

### Conclusiones y Recomendaciones

**Resumen del Análisis:**
Este notebook demostró el proceso completo de desarrollo de modelos de clasificación binaria para diagnóstico médico. Utilizando el dataset Breast Cancer Wisconsin, comparamos Regresión Logística y Árbol de Decisión, obteniendo métricas de evaluación comprehensivas.

**Lecciones Aprendidas:**
1. **Importancia del preprocesamiento**: La normalización mejora significativamente el rendimiento de algoritmos sensibles a la escala.
2. **Evaluación múltiple**: No confiar en una sola métrica; usar accuracy, precision, recall, specificity y AUC-ROC.
3. **Contexto del problema**: En aplicaciones médicas, el recall es más crítico que la precisión para evitar falsos negativos.
4. **Interpretabilidad**: Los árboles de decisión ofrecen mayor explicabilidad, crucial en medicina.

**Recomendaciones para Producción:**
- **Validación cruzada**: Usar k-fold cross-validation para evaluación más robusta.
- **Ajuste de hiperparámetros**: Optimizar parámetros con GridSearchCV o RandomizedSearchCV.
- **Ensemble methods**: Considerar Random Forest o Gradient Boosting para mejor rendimiento.
- **Validación externa**: Probar el modelo con datos de diferentes instituciones médicas.
- **Monitoreo continuo**: Los modelos médicos requieren actualización periódica con nuevos datos.

**Consideraciones Éticas:**
- Los modelos de ML en medicina deben ser interpretables y explicables.
- Siempre combinar con expertise médica; el modelo es una herramienta de apoyo, no un reemplazo.
- Considerar el costo de falsos positivos vs falsos negativos en el contexto específico.

Este ejercicio proporciona una base sólida para entender el desarrollo responsable de modelos de ML en aplicaciones críticas como el diagnóstico médico.

### Referencias y Recursos Adicionales

**Dataset:**
- Breast Cancer Wisconsin (Diagnostic) Dataset: https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic
- Documentación en scikit-learn: https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-dataset

**Conceptos de Machine Learning:**
- [Understanding ROC Curves](https://towardsdatascience.com/understanding-auc-roc-curve-68b2303cc9c5)
- [Precision vs Recall](https://towardsdatascience.com/precision-vs-recall-386d9c3a8d0d)
- [Logistic Regression Explained](https://towardsdatascience.com/logistic-regression-explained-9b7f2e0d4c6d)

**Librerías:**
- Scikit-learn User Guide: https://scikit-learn.org/stable/user_guide.html
- Pandas Documentation: https://pandas.pydata.org/docs/
- Matplotlib Tutorials: https://matplotlib.org/stable/tutorials/index.html

**Mejores Prácticas en ML Médico:**
- [Guidelines for ML in Healthcare](https://www.nature.com/articles/s41591-020-1048-6)
- [Interpretable ML](https://christophm.github.io/interpretable-ml-book/)

Este notebook sigue las mejores prácticas de desarrollo reproducible y bien documentado de modelos de machine learning.